# Evaluation Results (Tables 1 & 2 in Paper)

Please check the results on this notebook, as reported in our paper (Table 1 & 2). You can reproduce these results by requested the ckpt files from us.

Contact: zhanh.he.uwa@gmail.com

## Before you start - Setup the notebook Env path:

In [ ]:
%cd /media/datadisk/home/22828187/zhanh/202510_hpt_smc
%env PYTHONPATH=/media/datadisk/home/22828187/zhanh/202510_hpt_smc

## Evaluation on Maestro v3 Test Set (Table 1)

| AMT Module | Score Module | Input2 | Input3 | frame_max error ↓ | frame_max std ↓ | onset_masked error ↓ | onset_masked std ↓ | Recall_10% ↑ | Params |
|---|---|---|---|---:|---:|---:|---:|---:|---:|
| HPT | direct | – | – | 9.46 | 9.26 | 4.59 | 6.13 | 80.59% | 4.07 M |
| HPT | note editor | onset | – | 9.40 | 7.76 | 3.99 | 4.79 | 93.78% | 5.04 M |
| HPT | note editor | onset | frame | 9.02 | 7.64 | 4.00 | 4.70 | 93.88% | 5.04 M |
| HPT | note editor | onset | exframe | 9.55 | 7.68 | 3.91 | 4.71 | 94.04% | 5.04 M |

Because different methods has diff post-processing preference:
- Benchmark methods are reported with frame_max_error (MAE) and frame_max_std (STD)
- Proposed & AMT methods are reported with onset_masked_error (MAE) and onset_masked_std (STD)
- In this way, both are best score reported.

Besides,
- The Recall_10% here is used in my PhD Thesis, not in published paper. 
- Specifically, it comes from the mir_eval, different with the normal "Recall" used in FiLM-UNet paper (Kim and Serra, ismir2024).
- P/R/F1 from mir_eval is same, because we post-processing the VeloEst with the ground truth MIDI notes. No timing estimated errors, so P=R=F1.

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="maestro" \
  model.type="hpt" \
  score_informed.method="direct" \
  model.input2="null" model.input3="null" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hpt/100000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="maestro" \
  model.type="hpt" \
  score_informed.method="note_editor" \
  model.input2="onset" model.input3="null" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hpt+onset+score_note_editor/90000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="maestro" \
  model.type="hpt" \
  score_informed.method="note_editor" \
  model.input2="onset" model.input3="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hpt+onset+frame+score_note_editor/100000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="maestro" \
  model.type="hpt" \
  score_informed.method="note_editor" \
  model.input2="onset" model.input3="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hpt+onset+frame+score_note_editor/100000_iterations.pth"

## SMD Dataset Evaluation (Table2 in Paper)

| Category | Model | Variant / Setting | frame_error ↓ | frame_std ↓ | onset_error ↓ | onset_std ↓ | Recall_10% ↑ | Params |
|---|---|---|---:|---:|---:|---:|---:|---:|
| **Benchmark** | TransKun v2 | AMT (pretrained) | 13.27 | 11.90 | (13.95) | (12.28) | 75.13% in MIREX2024 (73.22%) | 13.61 M |
| **Benchmark** | FiLM-UNet | score-informed (pretrained) | 9.9 in their paper (9.95) | 7.8 in their paper (8.32) | 16.03 | 12.46 | 67.92% (careful, 89.7% in their paper is different metric) | 8.80 M |
| **Ours (trained)** | HPT | direct (null + null) | 11.69 | 9.13 | 10.00 | 8.79 | 62.48% | 4.07 M |
| **Ours (trained)** | HPT | score-inf (onset + frame) | 15.82 | 8.90 | 8.04 | 6.91 | 80.33% | 5.04 M |
| **Ours (trained)** | HPPNet | direct | 12.20 | 9.47 | 10.57 | 8.98 | 60.21% | 0.52 M |
| **Ours (trained)** | HPPNet | score-inf (onset + frame) | 16.81 | 9.40 | 9.17 | 7.31 | 79.25% | 1.48 M |
| **Ours (trained)** | DynEst | direct | 12.48 | 9.44 | 10.39 | 8.78 | 59.57% | 13.29 M |
| **Ours (trained)** | DynEst | score-inf (onset + frame) | 16.24 | 9.26 | 8.03 | 6.85 | 77.55% | 14.26 M |

Because different methods has diff post-processing preference:
- Benchmark methods are reported with frame_max_error (MAE) and frame_max_std (STD)
- Our methods are reported with onset_masked_error (MAE) and onset_masked_std (STD)
- In this way, both are best score reported.

Besides,
- The Recall_10% here is used in my PhD Thesis, not in published paper.
- FiLM-UNet MAE & STD: FiLM-UNet checkpoint (Kim and Serra, ISMIR 2024) reproduced results slightly deviated from their paper, we cite the original results from their paper, rather than using our own reproduction.
- FiLM-UNet Recall: has been reported 89.7% in their ISMIR2024 paper (any predicted velocity in the MIDI notes frame are counted as correct), which is less strict metric than our Recall_10% (when velocity valued within +-10% tolerance are counted as correct). 
- Transkun v2 (no augmentation) has reported 75.13% Recall_10% (Note On/Off with Velocity Metric in mir_eval) in [MIREX2024 Challenge](https://github.com/Yujia-Yan/Transkun). 


In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="transkun_pretrained" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/2025_achieve/202602_smc/other repos/Transkun/transkun/pretrained/2.0.pt" \
  feature.audio_feature="logmel" feature.sample_rate=44100 feature.frames_per_second=43


In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd"\
  model.type="filmunet_pretrained" \
  model.kim_condition="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_smc/other repos/kim_ismir2024/pretrained/FiLMUNetPretrained+frame/1000000_iterations.pth" \
  feature.audio_feature="logmel" feature.sample_rate=16000 feature.segment_seconds=2.0 feature.hop_seconds=1.0 feature.frames_per_second=100


In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd"\
  model.type="filmunet_pretrained" \
  model.kim_condition="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/2025_achieve/202602_smc/other repos/kim_ismir2024/pretrained/FiLMUNetPretrained+frame/1000000_iterations.pth" \
  feature.audio_feature="logmel" feature.sample_rate=16000 feature.segment_seconds=2.0 feature.hop_seconds=1.0 feature.frames_per_second=100


In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="hpt" \
  score_informed.method="direct" \
  model.input2="null" model.input3="null" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hpt/60000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="hpt" \
  score_informed.method="note_editor" \
  model.input2="onset" model.input3="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hpt+onset+frame+score_note_editor/20000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="hppnet" \
  score_informed.method="direct" \
  model.input2="null" model.input3="null" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hppnet/10000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="hppnet" \
  score_informed.method="note_editor" \
  model.input2="onset" model.input3="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/hppnet+onset+frame+score_note_editor/20000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="dynest" \
  score_informed.method="direct" \
  model.input2="null" model.input3="null" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/dynest/10000_iterations.pth"

In [ ]:
!python pytorch/calculate_scores.py --enable_mireval \
  exp.run_infer="single" dataset.test_set="smd" \
  model.type="dynest" \
  score_informed.method="note_editor" \
  model.input2="onset" model.input3="frame" \
  model.frontend_pretrained="/media/datadisk/home/22828187/zhanh/202510_hpt_data/workspaces/checkpoints/dynest+onset+frame+score_note_editor/10000_iterations.pth"

# Reserve Part to conduct experiment in Table 2

In [ ]:
!python pytorch/calculate_scores.py \
  exp.run_infer=multi \
  dataset.test_set=smd \
  model.type=hpt \
  model.input2=onset model.input3=frame \
  score_informed.method=note_editor \
  +exp.eval_start_iteration=0 +exp.eval_end_iteration=10000 +exp.eval_step_iteration=10000
  # Comment out the last line (DEBUG mode) in format - group search the best eval iteration, which is very time-consuming.